# Ciyaaro Category Cleaning Pipeline

This notebook cleans and prepares the **Ciyaaro** category dataset for the Somali headline generation + category prediction model.

Sources included:

- `goobjoog_ciyaaraha_scraped_articles_FIXED.xlsx`
- `laacibnet_scraped_articles.xlsx`

Final model format:

```text
input_text  = analyze somali news article: <body_clean>
target_text = category: ciyaaro | headline: <headline_clean>
```


In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)

GOOBJOOG_PATH = Path("raw_dataset/goobjoog_ciyaaraha_scraped_articles_FIXED.xlsx")
LAACIBNET_PATH = Path("raw_dataset/laacibnet_scraped_articles.xlsx")

OUTPUT_REVIEW_XLSX = Path("clean_dataset_for_review/ciyaaro_cleaned_for_review.xlsx")
OUTPUT_MODEL_CSV = Path("ready_dataset/ciyaaro_model_ready.csv")
OUTPUT_MODEL_XLSX = Path("ready_dataset/ciyaaro_model_ready.xlsx")
OUTPUT_DROPPED_REVIEW_XLSX = Path("for_dropped_dataset/ciyaaro_dropped_or_review_rows.xlsx")

CATEGORY_LABEL = "ciyaaro"
MIN_BODY_WORDS_FOR_MODEL = 50


## 1. Load the raw files

In [3]:
goobjoog = pd.read_excel(GOOBJOOG_PATH)
laacibnet = pd.read_excel(LAACIBNET_PATH)

print("Goobjoog shape:", goobjoog.shape)
print("Goobjoog columns:", goobjoog.columns.tolist())

print("\nLaacibnet shape:", laacibnet.shape)
print("Laacibnet columns:", laacibnet.columns.tolist())

display(goobjoog.head(3))
display(laacibnet.head(3))


Goobjoog shape: (674, 9)
Goobjoog columns: ['url', 'headline', 'body', 'category', 'source', 'status', 'body_word_count', 'scraped_at', 'error']

Laacibnet shape: (645, 5)
Laacibnet columns: ['url', 'headline', 'body', 'category', 'source']


,url,headline,body,category,source,status,body_word_count,scraped_at,error
0,https://goobjoog.com/2020/12/31/yaa-guulaysan-doono-gobolka-banaadir-iyo-galmudug-tartanka-maamul-goboleedyada-2020/,Yaa Guulaysan Doono Gobolka Banaadir Iyo Galmudug Tartanka Maamul Goboleedyada 2020,Kooxaha Kubbadda Cagta Gobolka Banaadir iyo Galmudug ayaa markii ugu horeesay wada dheeli doono heerka ugu danbeeya tartanka kubbadda cagta Maamul Goboleedyada iyo Gobolka Bana...,ciyaaro,Goobjoog,ok,224,2026-04-30 22:16:22,NaN
1,https://goobjoog.com/2020/10/25/xildhibaan-shaacir-guul-ayaan-u-rajaynaayaa-wasiir-xamsa-iyo-wasaarada-ciyaaraha-dalka/,Xildhibaan Shaacir “Guul Ayaan U Rajaynaayaa Wasiir Xamsa Iyo Wasaarada Ciyaaraha Dalka”,Xildhibaan Cabdullahi Maxamed Aadan (Shaacir) oo ka tirsan Xildhibaanada Golaha shacabka Baarlamaanka Soomaaliya horayna ugu mid ahaa xidigihii hore Soomaaliya ayaa u hambaleey...,ciyaaro,Goobjoog,ok,179,2026-04-30 22:16:25,NaN
2,https://goobjoog.com/2024/01/21/akhriso-tartanka-dowlad-goboleedyada-oo-dib-u-bilaabanaya-ogow-isku-aadka-iyo-waxkasta-oo-ku-saabsan-ila-aiyo-final-ka/,"Akhriso: Tartanka Dowlad Goboleedyada oo Dib u Bilaabanaya, Ogow isku aadka iyo Waxkasta oo Ku Saabsan Ila aiyo Final-ka","Garoonka Kubadda Cagta ee Stadium Muqdisho waxa maanta dib uga bilaabanaya tartanka dowlad goboleedyada iyo gobolka Banaadir, kaas oo maalmihii lasoo dhaafay hakad galay, kadib...",ciyaaro,Goobjoog,ok,319,2026-04-30 22:16:28,NaN


,url,headline,body,category,source
0,https://www.laacibnet.net/fa-ga-ingiriiska-oo-ganaax-culus-dul-dhigay-mid-kamid-ah-weeraryahannada-chelsea/,FA-ga Ingiriiska Oo Ganaax Culus Dul Dhigay Mid Kamid Ah Weeraryahannada Chelsea,"Xiriirka Kubadda Cagta ee dalka Ingiriiska (FA) ayaa maanta oo Arbaco ah shaaciyay go’aan naxdin leh oo ka dhan ah garabka kooxda Chelsea, Mykhailo Mudryk, kaas oo lagu riday g...",ciyaaraha_maanta,laacibnet
1,https://www.laacibnet.net/neymar-oo-shaaciyay-labada-dal-ee-uu-doonayo-inay-isugu-yimaadaan-final-ka-koobka-adduunka/,Neymar Oo Shaaciyay Labada Xul Ee Uu Doonayo Inay Isugu Yimaadaan Final-ka Koobka Adduunka,"Xiddiga caanka ah ee kooxda Santos iyo xulka qaranka Brazil, Neymar Jr, ayaa shaaca ka qaaday hammigiisa ku aaddan tartanka Koobka Adduunka ee 2026, kaas oo lagu wado inuu xaga...",ciyaaraha_maanta,laacibnet
2,https://www.laacibnet.net/thierry-henry-oo-kashifay-sirta-guulaha-arsenal-ee-xilli-ciyaareed-kaan-ka-hor-kulanka-atletico-madrid/,Thierry Henry Oo Kashifay Sirta Guulaha Arsenal Ee Xilli Ciyaareed Kaan Ka Hor Kulanka Atletico Madrid,"Halyeyga kooxda Arsenal, Thierry Henry, ayaa falanqayn qoto dheer ku sameeyay qaab ciyaareedka kooxdiisii hore ka hor kulanka adag ee semi-finalka tartanka Champions League-ga ...",ciyaaraha_maanta,laacibnet


## 2. Initial source-level analysis

This gives us the same checks we used for previous categories:

- missing values
- category/source values
- empty bodies/headlines
- duplicate URLs/headlines/bodies
- raw body length distribution


In [5]:
def analyze_raw_source(df, name):
    temp = df.copy()
    temp["body_str"] = temp["body"].fillna("").astype(str)
    temp["headline_str"] = temp["headline"].fillna("").astype(str)
    temp["body_word_count_raw"] = temp["body_str"].str.split().str.len()
    temp["headline_word_count_raw"] = temp["headline_str"].str.split().str.len()

    print("=" * 100)
    print(name)
    print("Shape:", temp.shape)
    print("\nMissing values:")
    print(temp.isna().sum())

    if "category" in temp.columns:
        print("\nCategory values:")
        print(temp["category"].value_counts(dropna=False))

    if "source" in temp.columns:
        print("\nSource values:")
        print(temp["source"].value_counts(dropna=False))

    if "status" in temp.columns:
        print("\nScrape status values:")
        print(temp["status"].value_counts(dropna=False))

    print("\nQuality counts:")
    print("Empty headline:", (temp["headline_str"].str.strip() == "").sum())
    print("Empty body:", (temp["body_str"].str.strip() == "").sum())
    print("Duplicate URLs:", temp["url"].duplicated().sum() if "url" in temp.columns else "No url column")
    print("Duplicate headlines:", temp["headline_str"].duplicated().sum())
    print("Duplicate bodies:", temp["body_str"].duplicated().sum())

    print("\nBody word count raw:")
    print(temp["body_word_count_raw"].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))

    print("\nVery short raw bodies under 50 words:")
    display(temp[temp["body_word_count_raw"] < 50][["url", "headline", "body_word_count_raw", "body"]].head(20))

analyze_raw_source(goobjoog, "GOOBJOOG RAW")
analyze_raw_source(laacibnet, "LAACIBNET RAW")


GOOBJOOG RAW
Shape: (674, 13)

Missing values:
url                          0
headline                     0
body                        13
category                     0
source                       0
status                       0
body_word_count              0
scraped_at                   0
error                      674
body_str                     0
headline_str                 0
body_word_count_raw          0
headline_word_count_raw      0
dtype: int64

Category values:
category
ciyaaro    674
Name: count, dtype: int64

Source values:
source
Goobjoog    674
Name: count, dtype: int64

Scrape status values:
status
ok                       661
headline_only_no_body     13
Name: count, dtype: int64

Quality counts:
Empty headline: 0
Empty body: 13
Duplicate URLs: 0
Duplicate headlines: 1
Duplicate bodies: 18

Body word count raw:
count     674.000000
mean      243.348665
std       198.728376
min         0.000000
1%          0.000000
5%         41.650000
10%        88.900000
25%      

,url,headline,body_word_count_raw,body
14,https://goobjoog.com/2020/06/13/yuusuf-yuusuf-wasiir-khadiijo-qalad-ayay-ku-haystaa-warbaahinta-goobjoog/,Yuusuf Yuusuf “ Wasiir Khadiijo Qalad Ayay Ku Haystaa Warbaahinta Goobjoog ”,49,Yuusuf Maxamed Yuusuf oo ah qoraa xaga ciyaaraha ayaa sheegay in Wasiirka ciyaaraha iyo dhallinyarada Dowladda Khadiijo Maxamed Diiriye iyo xiriirada ka dhisan Soomaaliya aysan...
20,https://goobjoog.com/2018/11/14/natiijo-maxay-kusoo-dhmaatay-ciyaartii-u-dhaxeeysay-dalalka-soomaaliya-iyo-ethiopia/,Natiijo:-Maxay Kusoo Dhmaatay Ciyaartii U Dhaxeeysay Dalalka Soomaaliya Iyo Ethiopia,0,NaN
40,https://goobjoog.com/2020/09/29/buurane-waa-ceyb-in-loo-gacan-qaadaa-garsoorayaasha-kubbadda-cagta/,Buurane “Waa Ceyb In Loo Gacan Qaadaa Garsoorayaasha Kubbadda Cagta”,41,"“Waa ceyb in loo gacan qaadaa garsoorsoorayaasha kubbadda cagta,Xiriirka kubbadda cagta waxaa laga rabaa in adkeeyaan sharciyadda lagu hagayo ciyaaryahanada iyo macallimiinta d..."
64,https://goobjoog.com/2015/01/12/warbixinta-ganacsiga-ee-sanadkii-2014-akhriso/,Warbixinta Ganacsiga ee Sanadkii 2014 (AKHRISO),39,"Waxaan halkaan idiin kugu soo gudbineynaa Warbixinta Sanadlaha ah Ee Ganacsiga.\nWarbixintaan ayaa ka hadli doonta 12-ka Bil ee Sanadkii Hore 2014, isbedalada ganacsi, gaar aha..."
66,https://goobjoog.com/2020/06/28/guddoomiye-saacid-degmada-warta-nabadda-taariikh-weyn-ayay-u-tahay-helitaanka-garoonka-stadium-muqdisho/,Guddoomiye Saacid “Degmada Warta Nabadda Taariikh Weyn Ayay U Tahay Helitaanka Garoonka Stadium Muqdisho ”,47,Guddoomiyaha Isboortiga Degmada Warta Nabada Saacid Xuseen Cabdi oo la hadlay Goobjoog ayaa sheegay in maamulka degmadiisa ay aad ugu faraxsanyihiin helitaanka garoonka ciyaara...
70,https://goobjoog.com/2020/06/16/ibraahim-geedow-dhismaha-stadium-muqdisho-waan-in-la-waa-fajiyaa-heerka-caalamka/,Ibraahim Geedow “Dhismaha Stadium Muqdisho Waan In La Waa Fajiyaa Heerka Caalamka ”,30,Ibraahim Geedow Cawaale oo ah Qoraa & falanqeeye ciyaaraha gudaha ayaa sheegay in dhismaha garoonka Stadium Muqdisho laga hoos marin heerka caalamiga ee garoomada Xiriirka Kubb...
73,https://goobjoog.com/2019/03/28/jubaland-maxaa-sababay-guuladaradii-kooxda-kaneva/,Jubaland-Maxaa Sababay Guuladaradii Kooxda Kaneva,33,Kulan ka mid ah kulamada heerka labaad ee kooxaha kubbadda Jubaland ayaa waxaa wada ciyaaray Shaqaalaha iyo Kaneva oo ka mid ah kuwa loo baqayo in sanadkan ay ku guulaystan hor...
79,https://goobjoog.com/2022/02/08/shirkadda-nike-oo-joojisay-wadashaqeyntii-ay-la-lahayd-weeraryahanka-kooxda-manchester-united-ee-mason-greenwood/,Shirkadda Nike Oo Joojisay Wadashaqeyntii Ay La Lahayd Weeraryahanka Kooxda Manchester United Ee Mason Greenwood,2,Goobjoog News
80,https://goobjoog.com/2019/03/21/garsoorayaal-soomaaliyeed-oo-kasoo-muuqday-is-reeb-reebka-afcon-2019/,Garsoorayaal Soomaaliyeed Oo Kasoo Muuqday Is Reeb Reebka Afcon U-23,0,NaN
81,https://goobjoog.com/2022/02/09/garoonka-camp-nou-oo-magaca-laga-beddelayo-shirkadda-la-wareegeysa-lacagaha-ay-ku-heli-doonto-kooxda-barcelona/,"Garoonka Camp Nou Oo Magaca Laga Beddelayo, Shirkadda La Wareegeysa & Lacagaha Ay Ku Heli Doonto Kooxda Barcelona",2,Goobjoog News


LAACIBNET RAW
Shape: (645, 9)

Missing values:
url                        0
headline                   0
body                       0
category                   0
source                     0
body_str                   0
headline_str               0
body_word_count_raw        0
headline_word_count_raw    0
dtype: int64

Category values:
category
ciyaaraha_maanta    645
Name: count, dtype: int64

Source values:
source
laacibnet    645
Name: count, dtype: int64

Quality counts:
Empty headline: 0
Empty body: 0
Duplicate URLs: 0
Duplicate headlines: 4
Duplicate bodies: 0

Body word count raw:
count    645.000000
mean     188.054264
std       46.663975
min       98.000000
1%       121.000000
5%       136.000000
10%      143.000000
25%      159.000000
50%      180.000000
75%      207.000000
90%      247.000000
95%      265.400000
99%      315.000000
max      619.000000
Name: body_word_count_raw, dtype: float64

Very short raw bodies under 50 words:


,url,headline,body_word_count_raw,body


## 3. Inspect common noise patterns

From analysis:

### Goobjoog
Common noise includes:

- `W/D-Maxamed Xuseen Qalinle`
- `W/D- Maxamed Xuseen Qalinle`
- `Goobjoog News`
- `Halkan Ka Daawo...`
- `Sidoo Kale Aqriso...`
- `Email- [email protected]`
- old `wordpress-...cloudwaysapps.com` links

### Laacibnet
Every article body contains:

- `Save my name, email, and website in this browser for the next time I comment.`
- `laacibnet`


In [8]:
def show_common_lines(df, name, top_n=30):
    counts = {}
    for text in df["body"].fillna("").astype(str):
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        for line in lines:
            counts[line] = counts.get(line, 0) + 1

    common = pd.DataFrame(
        sorted(counts.items(), key=lambda x: -x[1])[:top_n],
        columns=["line", "count"]
    )

    print("=" * 100)
    print(name)
    display(common)

show_common_lines(goobjoog, "Most common Goobjoog lines")
show_common_lines(laacibnet, "Most common Laacibnet lines")


Most common Goobjoog lines


,line,count
0,W/D-Maxamed Xuseen Qalinle,375
1,W/D- Maxamed Xuseen Qalinle,51
2,Goobjoog News,28
3,Sidoo Kale Aqriso Warar Ku Saabsan SOMFA,20
4,Maxamed Xuseen Qalinle,16
5,Ogeysiiska Kulanka,8
6,Halkan Ka Daawo Warbixin Ku Saabsan,7
7,Halkan Ka Daawo Qeyb Ka Mid Waraysi Dhinac Skyp-ka Uu Falanqeeye Geedow Siiyay Goobjoog,5
8,Email- [email protected],5
9,Garoonka- Banaadir Stadium,4


Most common Laacibnet lines


,line,count
0,"Save my name, email, and website in this browser for the next time I comment.",645
1,laacibnet,645
2,Wuxuu yiri:,21
3,Wuxuu intaas ku daray:,5
4,Shaxda la filayo (4-2-3-1):,3
5,Xaaladda ciyaartoyda:,3
6,Arsenal 2-0 Newcastle United,2
7,Robert Lewandowski,2
8,Marcus Rashford,2
9,Aston Villa,2


## 4. Cleaning functions

In [9]:
def normalize_basic_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\\n", "\n")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("\ufeff", "")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def clean_headline(text):
    text = normalize_basic_text(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


GOOBJOG_REMOVE_LINE_PATTERNS = [
    r"^Goobjoog News$",
    r"^Googjoog News$",
    r"^Dhageyso$",
    r"^Daawo$",
    r"^Halka[n]?\s+(ka\s+)?Daawo.*$",
    r"^Halka[n]?\s+hoose.*$",
    r"^Sidoo Kale Aqriso.*$",
    r"^Email\s*[:\-].*$",
    r"^\[email\s*protected\]$",
    r"^https?://.*$",
    r"^wordpress-.*$",
    r"^W/D\s*[-:].*$",
    r"^W/Q\s*[-:].*$",
    r"^Wariye\s*[-:].*$",
    r"^Maxamed Xuseen Qalinle$",
    r"^Xuseen Maxamed Hadaafow$",
    r"^Somali Football Federation Media Department$",
    r"^Waaxda warbaahinta.*$",
    r"^Wararka ciyaaraha Goobjoog news$",
    r"^Qoraalkan, waxa uu u gaar.*$",
    r"^Xogta Waxa ay gaar.*$",
]

LAACIBNET_REMOVE_LINE_PATTERNS = [
    r"^Save my name, email, and website in this browser for the next time I comment\.$",
    r"^laacibnet$",
]


def remove_noise_lines(text, source):
    text = normalize_basic_text(text)

    source_key = str(source).strip().lower()

    if source_key == "goobjoog":
        patterns = GOOBJOG_REMOVE_LINE_PATTERNS
    elif source_key == "laacibnet":
        patterns = LAACIBNET_REMOVE_LINE_PATTERNS
    else:
        patterns = []

    cleaned_lines = []

    for raw_line in text.splitlines():
        line = normalize_basic_text(raw_line)

        if not line:
            continue

        # Sometimes Goobjoog writer credits appear at the end of an otherwise useful line.
        # In that case, remove only the trailing writer-credit part, not the whole line.
        line = re.sub(r"\s+W/D\s*[-:].*$", "", line, flags=re.IGNORECASE).strip()
        line = re.sub(r"\s+W/Q\s*[-:].*$", "", line, flags=re.IGNORECASE).strip()

        if not line:
            continue

        should_remove = any(
            re.search(pattern, line, flags=re.IGNORECASE)
            for pattern in patterns
        )

        if not should_remove:
            cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    cleaned = re.sub(r"[ \t]+", " ", cleaned)

    return cleaned.strip()


## 5. Standardize source/category and apply cleaning

In [10]:
goobjoog_clean = goobjoog.copy()
laacibnet_clean = laacibnet.copy()

goobjoog_clean["source"] = "Goobjoog"
laacibnet_clean["source"] = "Laacibnet"

goobjoog_clean["category"] = CATEGORY_LABEL
laacibnet_clean["category"] = CATEGORY_LABEL

for df in [goobjoog_clean, laacibnet_clean]:
    df["headline_clean"] = df["headline"].apply(clean_headline)
    df["body_clean"] = df.apply(lambda row: remove_noise_lines(row["body"], row["source"]), axis=1)

    df["body_word_count_raw"] = df["body"].fillna("").astype(str).str.split().str.len()
    df["body_word_count_clean"] = df["body_clean"].str.split().str.len()

    df["headline_word_count_raw"] = df["headline"].fillna("").astype(str).str.split().str.len()
    df["headline_word_count_clean"] = df["headline_clean"].str.split().str.len()

print("Goobjoog cleaned shape:", goobjoog_clean.shape)
print("Laacibnet cleaned shape:", laacibnet_clean.shape)


Goobjoog cleaned shape: (674, 15)
Laacibnet cleaned shape: (645, 11)



## 6. Compare raw and clean body lengths

In [11]:
def compare_raw_clean_lengths(df, name):
    print("=" * 100)
    print(name)

    print("\nRaw body length:")
    print(df["body_word_count_raw"].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))

    print("\nClean body length:")
    print(df["body_word_count_clean"].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))

    print("\nEmpty clean bodies:", (df["body_clean"].str.strip() == "").sum())
    print("Clean bodies under 50 words:", (df["body_word_count_clean"] < 50).sum())
    print("Clean bodies under 100 words:", (df["body_word_count_clean"] < 100).sum())

compare_raw_clean_lengths(goobjoog_clean, "GOOBJOOG")
compare_raw_clean_lengths(laacibnet_clean, "LAACIBNET")


GOOBJOOG

Raw body length:
count     674.000000
mean      243.348665
std       198.728376
min         0.000000
1%          0.000000
5%         41.650000
10%        88.900000
25%       153.250000
50%       217.000000
75%       291.000000
90%       363.700000
95%       442.350000
99%      1184.820000
max      2293.000000
Name: body_word_count_raw, dtype: float64

Clean body length:
count     674.000000
mean      236.185460
std       197.182712
min         0.000000
1%          0.000000
5%         33.000000
10%        85.600000
25%       147.500000
50%       210.000000
75%       282.000000
90%       351.700000
95%       433.750000
99%      1163.690000
max      2286.000000
Name: body_word_count_clean, dtype: float64

Empty clean bodies: 24
Clean bodies under 50 words: 45
Clean bodies under 100 words: 86
LAACIBNET

Raw body length:
count    645.000000
mean     188.054264
std       46.663975
min       98.000000
1%       121.000000
5%       136.000000
10%      143.000000
25%      159.000000
50

## 7. Check remaining noise after cleaning

In [14]:
noise_checks = {
    "Goobjoog": [
        "Goobjoog News",
        "Googjoog News",
        "W/D-",
        "W/D:",
        "Halkan Ka Daawo",
        "Halkan Daawo",
        "HALKA HOOSE",
        "Sidoo Kale Aqriso",
        "wordpress-",
        "Save my name",
    ],
    "Laacibnet": [
        "Save my name",
        "laacibnet",
        "Goobjoog News",
        "Sidoo Kale Aqriso",
    ],
}

for name, df in [("Goobjoog", goobjoog_clean), ("Laacibnet", laacibnet_clean)]:
    print("=" * 100)
    print(name)
    for pattern in noise_checks[name]:
        count = df["body_clean"].str.contains(pattern, case=False, regex=False, na=False).sum()
        print(f"{pattern}: {count}")


Goobjoog
Goobjoog News: 7
Googjoog News: 0
W/D-: 0
W/D:: 0
Halkan Ka Daawo: 2
Halkan Daawo: 0
HALKA HOOSE: 0
Sidoo Kale Aqriso: 0
wordpress-: 0
Save my name: 0
Laacibnet
Save my name: 0
laacibnet: 0
Goobjoog News: 0
Sidoo Kale Aqriso: 0


## 8. Combine Ciyaaro sources

Now combine Goobjoog and Laacibnet after source-specific cleaning.


In [15]:
ciyaaro_all = pd.concat([goobjoog_clean, laacibnet_clean], ignore_index=True)

# Keep a stable column order where possible.
preferred_columns = [
    "url",
    "headline",
    "body",
    "category",
    "source",
    "status",
    "body_word_count",
    "scraped_at",
    "error",
    "headline_clean",
    "body_clean",
    "headline_word_count_raw",
    "headline_word_count_clean",
    "body_word_count_raw",
    "body_word_count_clean",
]

existing_columns = [col for col in preferred_columns if col in ciyaaro_all.columns]
other_columns = [col for col in ciyaaro_all.columns if col not in existing_columns]
ciyaaro_all = ciyaaro_all[existing_columns + other_columns]

print("Combined Ciyaaro shape:", ciyaaro_all.shape)
print("\nSource counts:")
print(ciyaaro_all["source"].value_counts())

print("\nCategory counts:")
print(ciyaaro_all["category"].value_counts())


Combined Ciyaaro shape: (1319, 15)

Source counts:
source
Goobjoog     674
Laacibnet    645
Name: count, dtype: int64

Category counts:
category
ciyaaro    1319
Name: count, dtype: int64


## 9. Duplicate and quality flags

In [16]:
ciyaaro_all["empty_headline_clean"] = ciyaaro_all["headline_clean"].fillna("").astype(str).str.strip().eq("")
ciyaaro_all["empty_body_clean"] = ciyaaro_all["body_clean"].fillna("").astype(str).str.strip().eq("")
ciyaaro_all["short_body_clean"] = ciyaaro_all["body_word_count_clean"] < MIN_BODY_WORDS_FOR_MODEL

ciyaaro_all["duplicate_url"] = ciyaaro_all["url"].duplicated(keep=False)
ciyaaro_all["duplicate_headline_clean"] = ciyaaro_all["headline_clean"].duplicated(keep=False)
ciyaaro_all["duplicate_body_clean"] = ciyaaro_all["body_clean"].duplicated(keep=False)

print("Total rows:", len(ciyaaro_all))
print("Empty clean headline:", ciyaaro_all["empty_headline_clean"].sum())
print("Empty clean body:", ciyaaro_all["empty_body_clean"].sum())
print(f"Short clean body under {MIN_BODY_WORDS_FOR_MODEL} words:", ciyaaro_all["short_body_clean"].sum())
print("Duplicate URL rows:", ciyaaro_all["duplicate_url"].sum())
print("Duplicate clean headline rows:", ciyaaro_all["duplicate_headline_clean"].sum())
print("Duplicate clean body rows:", ciyaaro_all["duplicate_body_clean"].sum())

print("\nRows that need manual review:")
review_mask = (
    ciyaaro_all["empty_headline_clean"] |
    ciyaaro_all["empty_body_clean"] |
    ciyaaro_all["short_body_clean"] |
    ciyaaro_all["duplicate_url"] |
    ciyaaro_all["duplicate_body_clean"]
)

display(ciyaaro_all.loc[
    review_mask,
    ["source", "url", "headline_clean", "body_word_count_raw", "body_word_count_clean", "body_clean"]
].head(50))


Total rows: 1319
Empty clean headline: 0
Empty clean body: 24
Short clean body under 50 words: 45
Duplicate URL rows: 0
Duplicate clean headline rows: 9
Duplicate clean body rows: 24

Rows that need manual review:


,source,url,headline_clean,body_word_count_raw,body_word_count_clean,body_clean
5,Goobjoog,https://goobjoog.com/2020/06/16/ibraahim-geedow-xiriirka-waa-in-uu-wakiilo-u-diraa-xirfadlayaasha-jooga-kenya/,"Ibraahim Geedow ""Xiriirka Kubbbadda CagtaS Soomaaliya Waa In Uu Wakiilo U Diraa Xirfadlayaasha Jooga Kenya """,57,43,Ibraahim Geedoow Cawaale oo ah qoraa iyo falanqeeye ciyaaraha gudaha ayaa sheegay Wadanka Kenya laga helayo ciyaartooy iyo macallimiin tayo wanaagsan kuwaas oo door wacan ka qa...
14,Goobjoog,https://goobjoog.com/2020/06/13/yuusuf-yuusuf-wasiir-khadiijo-qalad-ayay-ku-haystaa-warbaahinta-goobjoog/,"Yuusuf Yuusuf "" Wasiir Khadiijo Qalad Ayay Ku Haystaa Warbaahinta Goobjoog """,49,38,Yuusuf Maxamed Yuusuf oo ah qoraa xaga ciyaaraha ayaa sheegay in Wasiirka ciyaaraha iyo dhallinyarada Dowladda Khadiijo Maxamed Diiriye iyo xiriirada ka dhisan Soomaaliya aysan...
20,Goobjoog,https://goobjoog.com/2018/11/14/natiijo-maxay-kusoo-dhmaatay-ciyaartii-u-dhaxeeysay-dalalka-soomaaliya-iyo-ethiopia/,Natiijo:-Maxay Kusoo Dhmaatay Ciyaartii U Dhaxeeysay Dalalka Soomaaliya Iyo Ethiopia,0,0,
40,Goobjoog,https://goobjoog.com/2020/09/29/buurane-waa-ceyb-in-loo-gacan-qaadaa-garsoorayaasha-kubbadda-cagta/,"Buurane ""Waa Ceyb In Loo Gacan Qaadaa Garsoorayaasha Kubbadda Cagta""",41,41,"""Waa ceyb in loo gacan qaadaa garsoorsoorayaasha kubbadda cagta,Xiriirka kubbadda cagta waxaa laga rabaa in adkeeyaan sharciyadda lagu hagayo ciyaaryahanada iyo macallimiinta d..."
64,Goobjoog,https://goobjoog.com/2015/01/12/warbixinta-ganacsiga-ee-sanadkii-2014-akhriso/,Warbixinta Ganacsiga ee Sanadkii 2014 (AKHRISO),39,39,"Waxaan halkaan idiin kugu soo gudbineynaa Warbixinta Sanadlaha ah Ee Ganacsiga.\nWarbixintaan ayaa ka hadli doonta 12-ka Bil ee Sanadkii Hore 2014, isbedalada ganacsi, gaar aha..."
66,Goobjoog,https://goobjoog.com/2020/06/28/guddoomiye-saacid-degmada-warta-nabadda-taariikh-weyn-ayay-u-tahay-helitaanka-garoonka-stadium-muqdisho/,"Guddoomiye Saacid ""Degmada Warta Nabadda Taariikh Weyn Ayay U Tahay Helitaanka Garoonka Stadium Muqdisho """,47,37,Guddoomiyaha Isboortiga Degmada Warta Nabada Saacid Xuseen Cabdi oo la hadlay Goobjoog ayaa sheegay in maamulka degmadiisa ay aad ugu faraxsanyihiin helitaanka garoonka ciyaara...
70,Goobjoog,https://goobjoog.com/2020/06/16/ibraahim-geedow-dhismaha-stadium-muqdisho-waan-in-la-waa-fajiyaa-heerka-caalamka/,"Ibraahim Geedow ""Dhismaha Stadium Muqdisho Waan In La Waa Fajiyaa Heerka Caalamka """,30,30,Ibraahim Geedow Cawaale oo ah Qoraa & falanqeeye ciyaaraha gudaha ayaa sheegay in dhismaha garoonka Stadium Muqdisho laga hoos marin heerka caalamiga ee garoomada Xiriirka Kubb...
73,Goobjoog,https://goobjoog.com/2019/03/28/jubaland-maxaa-sababay-guuladaradii-kooxda-kaneva/,Jubaland-Maxaa Sababay Guuladaradii Kooxda Kaneva,33,33,Kulan ka mid ah kulamada heerka labaad ee kooxaha kubbadda Jubaland ayaa waxaa wada ciyaaray Shaqaalaha iyo Kaneva oo ka mid ah kuwa loo baqayo in sanadkan ay ku guulaystan hor...
79,Goobjoog,https://goobjoog.com/2022/02/08/shirkadda-nike-oo-joojisay-wadashaqeyntii-ay-la-lahayd-weeraryahanka-kooxda-manchester-united-ee-mason-greenwood/,Shirkadda Nike Oo Joojisay Wadashaqeyntii Ay La Lahayd Weeraryahanka Kooxda Manchester United Ee Mason Greenwood,2,0,
80,Goobjoog,https://goobjoog.com/2019/03/21/garsoorayaal-soomaaliyeed-oo-kasoo-muuqday-is-reeb-reebka-afcon-2019/,Garsoorayaal Soomaaliyeed Oo Kasoo Muuqday Is Reeb Reebka Afcon U-23,0,0,


## 10. Create model-ready dataset

Rules used here:

1. Remove empty clean headline.
2. Remove empty clean body.
3. Remove exact duplicate URLs.
4. Remove exact duplicate clean bodies.
5. Exclude clean bodies under 50 words from the model-ready file.
6. Keep duplicate headlines, but flag them in the review file, because some sports articles can naturally share very similar headline templates.


In [17]:
dropped_or_review = ciyaaro_all[
    ciyaaro_all["empty_headline_clean"] |
    ciyaaro_all["empty_body_clean"] |
    ciyaaro_all["short_body_clean"] |
    ciyaaro_all["duplicate_url"] |
    ciyaaro_all["duplicate_body_clean"] |
    ciyaaro_all["duplicate_headline_clean"]
].copy()

model_ready = ciyaaro_all.copy()

before = len(model_ready)
model_ready = model_ready[
    (~model_ready["empty_headline_clean"]) &
    (~model_ready["empty_body_clean"])
].copy()
after_empty = len(model_ready)

model_ready = model_ready.drop_duplicates(subset=["url"], keep="first")
after_url = len(model_ready)

model_ready = model_ready.drop_duplicates(subset=["body_clean"], keep="first")
after_body = len(model_ready)

model_ready = model_ready[model_ready["body_word_count_clean"] >= MIN_BODY_WORDS_FOR_MODEL].copy()
after_short = len(model_ready)

model_ready["input_text"] = "analyze somali news article: " + model_ready["body_clean"]
model_ready["target_text"] = "category: " + CATEGORY_LABEL + " | headline: " + model_ready["headline_clean"]

print("Rows before filtering:", before)
print("After removing empty headline/body:", after_empty)
print("After dropping duplicate URLs:", after_url)
print("After dropping duplicate clean bodies:", after_body)
print(f"After excluding bodies under {MIN_BODY_WORDS_FOR_MODEL} words:", after_short)

print("\nFinal source counts:")
print(model_ready["source"].value_counts())

print("\nFinal category counts:")
print(model_ready["category"].value_counts())

print("\nRemaining duplicate clean headlines:", model_ready["headline_clean"].duplicated().sum())

display(model_ready[["source", "url", "headline_clean", "body_word_count_clean", "input_text", "target_text"]].head(10))


Rows before filtering: 1319
After removing empty headline/body: 1295
After dropping duplicate URLs: 1295
After dropping duplicate clean bodies: 1295
After excluding bodies under 50 words: 1274

Final source counts:
source
Laacibnet    645
Goobjoog     629
Name: count, dtype: int64

Final category counts:
category
ciyaaro    1274
Name: count, dtype: int64

Remaining duplicate clean headlines: 5


,source,url,headline_clean,body_word_count_clean,input_text,target_text
0,Goobjoog,https://goobjoog.com/2020/12/31/yaa-guulaysan-doono-gobolka-banaadir-iyo-galmudug-tartanka-maamul-goboleedyada-2020/,Yaa Guulaysan Doono Gobolka Banaadir Iyo Galmudug Tartanka Maamul Goboleedyada 2020,221,analyze somali news article: Kooxaha Kubbadda Cagta Gobolka Banaadir iyo Galmudug ayaa markii ugu horeesay wada dheeli doono heerka ugu danbeeya tartanka kubbadda cagta Maamul ...,category: ciyaaro | headline: Yaa Guulaysan Doono Gobolka Banaadir Iyo Galmudug Tartanka Maamul Goboleedyada 2020
1,Goobjoog,https://goobjoog.com/2020/10/25/xildhibaan-shaacir-guul-ayaan-u-rajaynaayaa-wasiir-xamsa-iyo-wasaarada-ciyaaraha-dalka/,"Xildhibaan Shaacir ""Guul Ayaan U Rajaynaayaa Wasiir Xamsa Iyo Wasaarada Ciyaaraha Dalka""",176,analyze somali news article: Xildhibaan Cabdullahi Maxamed Aadan (Shaacir) oo ka tirsan Xildhibaanada Golaha shacabka Baarlamaanka Soomaaliya horayna ugu mid ahaa xidigihii hor...,"category: ciyaaro | headline: Xildhibaan Shaacir ""Guul Ayaan U Rajaynaayaa Wasiir Xamsa Iyo Wasaarada Ciyaaraha Dalka"""
2,Goobjoog,https://goobjoog.com/2024/01/21/akhriso-tartanka-dowlad-goboleedyada-oo-dib-u-bilaabanaya-ogow-isku-aadka-iyo-waxkasta-oo-ku-saabsan-ila-aiyo-final-ka/,"Akhriso: Tartanka Dowlad Goboleedyada oo Dib u Bilaabanaya, Ogow isku aadka iyo Waxkasta oo Ku Saabsan Ila aiyo Final-ka",319,"analyze somali news article: Garoonka Kubadda Cagta ee Stadium Muqdisho waxa maanta dib uga bilaabanaya tartanka dowlad goboleedyada iyo gobolka Banaadir, kaas oo maalmihii las...","category: ciyaaro | headline: Akhriso: Tartanka Dowlad Goboleedyada oo Dib u Bilaabanaya, Ogow isku aadka iyo Waxkasta oo Ku Saabsan Ila aiyo Final-ka"
3,Goobjoog,https://goobjoog.com/2020/07/11/maxamuud-maxamed-waxaan-diyaar-u-nahay-horomarka-iyo-kor-u-qaadida-ciyaaraha-degmada-diinsoor/,"Maxamuud Maxamed ""Waxaan Diyaar U Nahay Horomarka Iyo Kor U Qaadida Ciyaaraha Diinsoor """,196,"analyze somali news article: Maamulka Ciyaaraha Degmada Diinsoor ayaa tartan u qabtay kooxaha ka dhisan degmadaas,iyagoo ugu magac daray is dhaxglka bulshada iyo nabadda degmad...","category: ciyaaro | headline: Maxamuud Maxamed ""Waxaan Diyaar U Nahay Horomarka Iyo Kor U Qaadida Ciyaaraha Diinsoor """
4,Goobjoog,https://goobjoog.com/2019/06/10/wax-badan-ka-ogaw-kulanka-horseed-vs-muqdisho-city-club/,Wax Badan Ka Ogaw Kulanka Horseed VS Muqdisho City Club,317,analyze somali news article: Kooxda Ciidanka xooda dalka ee Horseed ayaa wajhaysa dhigeeda Muqdisho City Club oo sanadkii lasoo dhaafay ay wada ciyaareen kama dambeysta tartank...,category: ciyaaro | headline: Wax Badan Ka Ogaw Kulanka Horseed VS Muqdisho City Club
6,Goobjoog,https://goobjoog.com/2020/08/29/daadir-amiin-waan-ku-faraxsanahay-hanashada-horyaalka-heerka-labaad-ee-soomaaliya/,"Daadir Amiin ""Waan Ku Faraxsanahay Hanashada Horyaalka Heerka Koowaad Ee Soomaaliya """,107,analyze somali news article: Kooxda Geeska Afrika ayaa ku guulaystay horyaalka heerka ee kubbadda cagta Sooamaliya 2019-2020 waxaana guusha u qaaday tababarahooda Daadir Amiin ...,"category: ciyaaro | headline: Daadir Amiin ""Waan Ku Faraxsanahay Hanashada Horyaalka Heerka Koowaad Ee Soomaaliya """
7,Goobjoog,https://goobjoog.com/2019/03/28/hordhaca-horyaalka-somaaliya-yaa-guulo-badan-kooxaha-heegan-vs-dekedda/,Hordhaca Horyaalka Somaaliya-Yaa Guulo Badan Kooxaha Heegan VS Dekedda,520,analyze somali news article: Kooxda ciidan booliska Soomaaliyeed ee Heegan ayaa soo dhawaynaysa dhigeeda Dekedda oo ah naadiga difcaanaysa horyaalka labadii xili ciyaareed laso...,category: ciyaaro | headline: Hordhaca Horyaalka Somaaliya-Yaa Guulo Badan Kooxaha Heegan VS Dekedda
8,Goobjoog,https://goobjoog.com/2019/11/21/daaru-tarbiya-oo-ku-guulaystay-horyaalka-tartanka-iskuulka-2019/,Daaru Tarbiya Oo Ku Guulaystay Tartanka Iskuulada,225,"analyze somali news article: Waxaa lasoo gaba-gabeeyay koobkii kubbadda cagta dhallinyarada ee sanadka 2019-ka, waxaana ku guuleysatay kooxda 

## 11. Final checks before saving

In [18]:
print("All category values:")
print(model_ready["category"].value_counts(dropna=False))

print("\nMissing values in key columns:")
print(model_ready[["url", "headline_clean", "body_clean", "category", "source", "input_text", "target_text"]].isna().sum())

print("\nEmpty key text fields:")
print("empty headline_clean:", model_ready["headline_clean"].str.strip().eq("").sum())
print("empty body_clean:", model_ready["body_clean"].str.strip().eq("").sum())
print("empty input_text:", model_ready["input_text"].str.strip().eq("").sum())
print("empty target_text:", model_ready["target_text"].str.strip().eq("").sum())

print("\nDuplicate checks:")
print("duplicate url:", model_ready["url"].duplicated().sum())
print("duplicate body_clean:", model_ready["body_clean"].duplicated().sum())

print("\nLength checks:")
print(model_ready["body_word_count_clean"].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))


All category values:
category
ciyaaro    1274
Name: count, dtype: int64

Missing values in key columns:
url               0
headline_clean    0
body_clean        0
category          0
source            0
input_text        0
target_text       0
dtype: int64

Empty key text fields:
empty headline_clean: 0
empty body_clean: 0
empty input_text: 0
empty target_text: 0

Duplicate checks:
duplicate url: 0
duplicate body_clean: 0

Length checks:
count    1274.000000
mean      211.497645
std       146.302999
min        51.000000
1%         73.190000
5%        108.650000
10%       123.000000
25%       147.000000
50%       179.000000
75%       235.000000
90%       307.000000
95%       361.350000
99%       761.850000
max      2286.000000
Name: body_word_count_clean, dtype: float64


## 12. Save outputs

In [19]:
ciyaaro_all.to_excel(OUTPUT_REVIEW_XLSX, index=False)
model_ready.to_csv(OUTPUT_MODEL_CSV, index=False, encoding="utf-8-sig")
model_ready.to_excel(OUTPUT_MODEL_XLSX, index=False)
dropped_or_review.to_excel(OUTPUT_DROPPED_REVIEW_XLSX, index=False)

print("Saved:")
print(OUTPUT_REVIEW_XLSX)
print(OUTPUT_MODEL_CSV)
print(OUTPUT_MODEL_XLSX)
print(OUTPUT_DROPPED_REVIEW_XLSX)


Saved:
clean_dataset_for_review\ciyaaro_cleaned_for_review.xlsx
ready_dataset\ciyaaro_model_ready.csv
ready_dataset\ciyaaro_model_ready.xlsx
for_dropped_dataset\ciyaaro_dropped_or_review_rows.xlsx
